# AOT Analyzer — ML Model Training
## Gold XAU/USD 15M Direction Predictor

### Method 1 (recommended):
1. Open Colab file browser (left sidebar → folder icon → upload icon)
2. Upload `gold_15m_features.csv` from your PC
3. Run all cells below

### Method 2:
1. Run cell 2 — click 'Choose Files' to select the CSV
2. Run all remaining cells

### Output:
- Download `model_weights.json` → place in `web/public/ml/model_weights.json`

In [ ]:
import numpy as np
import pandas as pd
import json
import io
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix)
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

In [ ]:
# Try loading the CSV — first check if already in filesystem (manual upload),
# then fall back to files.upload()
csv_filename = 'gold_15m_features.csv'

if os.path.exists(csv_filename):
    print(f'Found existing file: {csv_filename}')
    df = pd.read_csv(csv_filename)
else:
    print(f'File not found in filesystem. Opening upload dialog...')
    from google.colab import files
    uploaded = files.upload()
    fn = list(uploaded.keys())[0]
    raw = uploaded[fn]
    # Handle both bytes and string content
    if isinstance(raw, bytes):
        df = pd.read_csv(io.BytesIO(raw))
    else:
        df = pd.read_csv(io.StringIO(raw))
    print(f'Loaded via upload: {fn}')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Separate features and target
drop_cols = ['time', 'open', 'high', 'low', 'close', 'target']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df['target'].values

# Handle missing values
X = X.fillna(X.median())

print(f'Features: {len(feature_cols)}')
print(f'Samples: {len(X)}')
print(f'Target balance: UP={y.sum()}, DOWN={len(y)-y.sum()} ({y.mean()*100:.1f}% UP)')

In [ ]:
# Train/val/test split (chronological order maintained)
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
y_train = y[:train_end]
X_val = X.iloc[train_end:val_end]
y_val = y[train_end:val_end]
X_test = X.iloc[val_end:]
y_test = y[val_end:]

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.values)
X_val_s = scaler.transform(X_val.values)
X_test_s = scaler.transform(X_test.values)

# Handle class imbalance
class_weights = class_weight.compute_class_weight(
    'balanced', classes=np.array([0, 1]), y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f'Class weights: {class_weight_dict}')

In [ ]:
# Build model (small, fast inference)
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_s.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

model.summary()

In [ ]:
# Early stopping to prevent overfitting
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluate — find optimal threshold for highest precision (win rate)
y_prob = model.predict(X_test_s).flatten()

thresholds = np.arange(0.5, 0.99, 0.01)
best_threshold = 0.5
best_wr = 0

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        wr = tp / (tp + fp) if (tp + fp) > 0 else 0
        if wr > best_wr and (tp + fp) >= 20:
            best_wr = wr
            best_threshold = thresh

print(f'Best threshold for WR: {best_threshold:.2f}')
print(f'Win rate (precision) at threshold: {best_wr*100:.1f}%')
print(f'Signals at threshold: {int((y_prob >= best_threshold).sum())}')

# Final confusion matrix
y_pred_final = (y_prob >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_final)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    total_pred = tp + fp
    print(f'\nTrue UP: {tp}, False UP (losses): {fp}')
    print(f'WIN RATE: {tp/(tp+fp)*100:.1f}%' if total_pred > 0 else 'No UP predictions')

In [ ]:
# Export model weights for TypeScript inference
def extract_weights(tf_model):
    layers_out = []
    for layer in tf_model.layers:
        if isinstance(layer, layers.Dense):
            w, b = layer.get_weights()
            activation = layer.activation.__name__
            layers_out.append({
                'weights': w.tolist(),
                'bias': b.tolist(),
                'activation': 'relu' if 'relu' in activation else 'sigmoid'
            })
    return layers_out

y_prob_all = model.predict(X_test_s).flatten()
y_pred_all = (y_prob_all >= best_threshold).astype(int)

acc = accuracy_score(y_test, y_pred_all)
prec = precision_score(y_test, y_pred_all, zero_division=0)
rec = recall_score(y_test, y_pred_all, zero_division=0)

model_json = {
    'feature_names': feature_cols,
    'mean': scaler.mean_.tolist(),
    'std': np.sqrt(scaler.var_).tolist(),
    'layers': extract_weights(model),
    'metadata': {
        'version': '1.0.0',
        'trained_on': str(pd.Timestamp.now()),
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'win_rate_at_threshold': float(best_wr),
        'decision_threshold': float(best_threshold),
    }
}

with open('model_weights.json', 'w') as f:
    json.dump(model_json, f, indent=2)

print(f'Model weights exported to model_weights.json')
print(f'\nModel metadata:')
for k, v in model_json['metadata'].items():
    print(f'  {k}: {v}')

In [ ]:
# Download the model weights
from google.colab import files
files.download('model_weights.json')

### Done!
Place the downloaded `model_weights.json` into `web/public/ml/model_weights.json`

Then restart the dev server — the ML model will load on the first signal request.